# PRISM Colab Runner

This notebook clones PRISM from GitHub, installs it in editable mode, runs smoke checks, generates a small ToyFireEnv dataset, trains a debug model, evaluates prediction metrics, and displays the generated figures.

Recommended Colab runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# ===== User settings =====
REPO_URL = "https://github.com/KKKKKK-y/prism.git"
BRANCH = "main"

# If the repository is private, paste a GitHub token here temporarily.
# Leave empty for a public repository. Do not print this value.
GITHUB_TOKEN = ""

PROJECT_DIR = "/content/prism"

# Debug run.
DEBUG_TRAIN_EPISODES = 10
DEBUG_VAL_EPISODES = 3
DEBUG_TEST_EPISODES = 3
DEBUG_EPOCHS = 1

# Level A full training for Colab T4 standard RAM.
RUN_FULL_TRAINING = True
RUN_FORMAL_PIPELINE = RUN_FULL_TRAINING
RUN_STAGE5_BASELINES = True
BACKUP_TO_DRIVE = True

TRAIN_EPISODES = 100
VAL_EPISODES = 20
TEST_EPISODES = 20
EPOCHS = 50
EVAL_EPISODES = 50
BASELINE_EPISODES = 50

# Backward-compatible aliases used by the existing formal pipeline cell.
FORMAL_TRAIN_EPISODES = TRAIN_EPISODES
FORMAL_VAL_EPISODES = VAL_EPISODES
FORMAL_TEST_EPISODES = TEST_EPISODES
FORMAL_EPOCHS = EPOCHS
FORMAL_EVAL_EPISODES = EVAL_EPISODES

# Level B, run only after Level A succeeds and you decide to continue:
# train=120, val=24, test=24, epochs=50, eval=75.
# Level C may exceed standard Colab RAM: train=150, val=30, test=30, epochs=50, eval=100.

DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/PRISM_runs"
DRIVE_RUN_DIR = None


In [ ]:
import getpass
import os
import shutil
import subprocess
from pathlib import Path

def run(cmd, cwd=None, check=True):
    safe_cmd = ["<TOKEN>" if GITHUB_TOKEN and part == GITHUB_TOKEN else part for part in cmd] if isinstance(cmd, list) else cmd
    print("\n$", " ".join(safe_cmd) if isinstance(safe_cmd, list) else safe_cmd)
    env = os.environ.copy()
    project_parent = str(Path(PROJECT_DIR).expanduser().resolve().parent)
    env["PYTHONPATH"] = project_parent + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    return subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str), check=check, env=env)

def tokenized_url(repo_url, token):
    return repo_url.replace("https://", f"https://{token}@")

if Path(PROJECT_DIR).exists():
    shutil.rmtree(PROJECT_DIR)

clone_result = run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=False)
if clone_result.returncode != 0:
    print("Public clone failed. If this is a private repo, paste a GitHub token with repo read access.")
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass.getpass("GitHub token: ")
    run(["git", "clone", "--branch", BRANCH, tokenized_url(REPO_URL, GITHUB_TOKEN), PROJECT_DIR])

os.chdir(PROJECT_DIR)
print("Project:", Path.cwd())
run(["git", "pull", "--ff-only"])
run(["git", "rev-parse", "--short", "HEAD"])
run(["git", "log", "-1", "--oneline"])

In [ ]:
# Install PRISM. Colab usually already includes a CUDA-enabled PyTorch build.
run(["python", "-m", "pip", "install", "-U", "pip"])
run(["python", "-m", "pip", "install", "-e", "."])

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
# Resource detection and optional Google Drive backup setup.
import os
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

import torch

os.chdir(PROJECT_DIR)

def _meminfo_gb():
    values = {}
    try:
        with open('/proc/meminfo', 'r', encoding='utf-8') as f:
            for line in f:
                key, value = line.split(':', 1)
                if key in {'MemTotal', 'MemAvailable'}:
                    values[key] = int(value.strip().split()[0]) / (1024 * 1024)
    except Exception:
        pass
    return values

print('===== Colab resource check =====')
print('torch.cuda.is_available():', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(f'GPU memory free/total GB: {free_bytes / 1024**3:.2f} / {total_bytes / 1024**3:.2f}')
    subprocess.run(['nvidia-smi'], check=False)
mem = _meminfo_gb()
print(f"CPU RAM total GB: {mem.get('MemTotal', float('nan')):.2f}")
print(f"CPU RAM available GB: {mem.get('MemAvailable', float('nan')):.2f}")
print('current working directory:', Path.cwd())
print('current git commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('latest git log:', subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True).strip())

if RUN_FULL_TRAINING and not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Stop formal training and switch Colab runtime to GPU.')

if BACKUP_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        DRIVE_RUN_DIR = Path(DRIVE_BACKUP_ROOT) / f'run_{timestamp}'
        DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
        print('Google Drive mounted: True')
        print('Drive backup directory:', DRIVE_RUN_DIR)
    except Exception as exc:
        DRIVE_RUN_DIR = None
        print('Google Drive mounted: False')
        print('Drive mount failed; continuing local Colab run:', repr(exc))
else:
    print('Google Drive backup disabled.')

def backup_to_drive(label=''):
    if not BACKUP_TO_DRIVE or DRIVE_RUN_DIR is None:
        print('Drive backup skipped:', label or 'no label')
        return
    items = [
        Path('outputs/checkpoints_toy/best.pt'),
        Path('outputs/checkpoints_toy/last.pt'),
        Path('outputs/results'),
        Path('outputs/visualizations'),
        Path('outputs/prism_formal_results.zip'),
        Path('outputs/prism_stage5_results.zip'),
        Path('configs/toy_train.yaml'),
        Path('outputs/results/formal_run_summary.txt'),
        Path('outputs/results/stage5_baseline_results.csv'),
    ]
    print('Backing up to Google Drive:', DRIVE_RUN_DIR, label)
    for item in items:
        if not item.exists():
            print('MISSING:', item)
            continue
        destination = DRIVE_RUN_DIR / item
        destination.parent.mkdir(parents=True, exist_ok=True)
        if item.is_dir():
            shutil.copytree(item, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(item, destination)
        size = sum(p.stat().st_size for p in item.rglob('*') if p.is_file()) if item.is_dir() else item.stat().st_size
        print(f'FOUND: {item} -> {destination} ({size / 1024**2:.2f} MB)')


In [ ]:
# Quick project checks.
run(["python", "scripts/check_project_ready.py"])
run(["python", "scripts/test_env.py"])
run(["python", "scripts/test_shapes.py", "--config", "configs/smoke.yaml"])
run(["python", "scripts/test_planner.py", "--config", "configs/smoke.yaml"])

In [ ]:
# Generate a debug ToyFireEnv dataset for a fast end-to-end Colab run.
run([
    "python", "scripts/generate_toy_dataset.py",
    "--config", "configs/toy_train.yaml",
    "--train_episodes", str(DEBUG_TRAIN_EPISODES),
    "--val_episodes", str(DEBUG_VAL_EPISODES),
    "--test_episodes", str(DEBUG_TEST_EPISODES),
])
run(["python", "scripts/test_toy_dataset.py", "--npz", "outputs/datasets/toy_fire_train.npz"])

In [ ]:
# Debug training with limited samples. This should finish quickly.
run(["python", "scripts/train.py", "--config", "configs/toy_train.yaml", "--epochs", str(DEBUG_EPOCHS), "--debug"])

In [ ]:
# Evaluate predictions and create figures.
run([
    "python", "scripts/evaluate_prediction_on_toy.py",
    "--config", "configs/toy_train.yaml",
    "--checkpoint", "outputs/checkpoints_toy/best.pt",
])
run([
    "python", "scripts/plot_training_curve.py",
    "--csv", "outputs/results/stage4_toy_training_log.csv",
    "--output", "outputs/visualizations/stage4_toy_training_curve.png",
])
run([
    "python", "scripts/plot_prediction_metrics.py",
    "--csv", "outputs/results/stage4_toy_prediction_metrics.csv",
    "--output", "outputs/visualizations/stage4_toy_prediction_metrics.png",
])
run([
    "python", "scripts/visualize_prediction.py",
    "--config", "configs/toy_train.yaml",
    "--checkpoint", "outputs/checkpoints_toy/best.pt",
    "--output", "outputs/visualizations/stage4_toy_prediction_horizons.png",
    "--all_horizons",
])

In [ ]:
# Display generated outputs.
from IPython.display import Image, display

for image_path in [
    "outputs/visualizations/stage4_toy_training_curve.png",
    "outputs/visualizations/stage4_toy_prediction_metrics.png",
    "outputs/visualizations/stage4_toy_prediction_horizons.png",
]:
    path = Path(image_path)
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)

In [ ]:
# Formal pipeline. Set RUN_FORMAL_PIPELINE = False in the first cell for debug-only runs.
if RUN_FORMAL_PIPELINE:
    import os
    import shutil
    import subprocess
    from pathlib import Path

    import torch

    os.chdir(PROJECT_DIR)
    current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    current_log = subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip()
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"

    print("Formal run summary before execution")
    print("RUN_FORMAL_PIPELINE:", RUN_FORMAL_PIPELINE)
    print("FORMAL_TRAIN_EPISODES:", FORMAL_TRAIN_EPISODES)
    print("FORMAL_VAL_EPISODES:", FORMAL_VAL_EPISODES)
    print("FORMAL_TEST_EPISODES:", FORMAL_TEST_EPISODES)
    print("FORMAL_EPOCHS:", FORMAL_EPOCHS)
    print("FORMAL_EVAL_EPISODES:", FORMAL_EVAL_EPISODES)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    print("GPU name:", gpu_name)
    print("current working directory:", Path.cwd())
    print("current git commit:", current_commit)
    print("latest git log:", current_log)
    if not torch.cuda.is_available():
        print("WARNING: CUDA is not available. Formal training may be very slow.")

    run(["python", "scripts/check_project_ready.py"])
    run(["python", "scripts/test_env.py"])
    run(["python", "scripts/test_shapes.py", "--config", "configs/smoke.yaml"])
    run(["python", "scripts/test_uncertainty.py", "--config", "configs/smoke.yaml", "--checkpoint", "outputs/checkpoints/best.pt"])
    run(["python", "scripts/test_planner.py", "--config", "configs/smoke.yaml"])

    run([
        "python", "scripts/generate_toy_dataset.py",
        "--config", "configs/toy_train.yaml",
        "--train_episodes", str(FORMAL_TRAIN_EPISODES),
        "--val_episodes", str(FORMAL_VAL_EPISODES),
        "--test_episodes", str(FORMAL_TEST_EPISODES),
    ])
    run(["python", "scripts/test_toy_dataset.py", "--npz", "outputs/datasets/toy_fire_train.npz"])

    run([
        "python", "scripts/run_stage4_3_pipeline.py",
        "--config", "configs/toy_train.yaml",
        "--epochs", str(FORMAL_EPOCHS),
        "--closed-loop-episodes", str(FORMAL_EVAL_EPISODES),
    ])
    run(["python", "scripts/package_results.py", "--output", "outputs/prism_formal_results.zip"])

    print("\n===== formal_run_summary.txt =====")
    run(["cat", "outputs/results/formal_run_summary.txt"], check=False)

    print("\n===== generated files =====")
    run(["ls", "-lh", "outputs/"], check=False)
    run(["ls", "-lh", "outputs/results/"], check=False)
    run(["ls", "-lh", "outputs/visualizations/"], check=False)
    run(["ls", "-lh", "outputs/checkpoints_toy/"], check=False)

    print("\n===== required output check =====")
    required_outputs = [
        "outputs/prism_formal_results.zip",
        "outputs/results/formal_run_summary.txt",
        "outputs/checkpoints_toy/best.pt",
        "outputs/checkpoints_toy/last.pt",
        "outputs/results/stage4_toy_training_log.csv",
        "outputs/results/stage4_toy_prediction_metrics.csv",
        "outputs/results/stage4_closed_loop_results.csv",
        "outputs/visualizations/stage4_toy_training_curve.png",
        "outputs/visualizations/stage4_toy_prediction_metrics.png",
        "outputs/visualizations/stage4_toy_prediction_horizons.png",
    ]
    for output in required_outputs:
        path = Path(output)
        print(("FOUND: " if path.exists() else "MISSING: ") + output)

    zip_path = Path("outputs/prism_formal_results.zip")
    backup_to_drive("after formal pipeline")

    print("Results zip:", zip_path)
    print("Download outputs/prism_formal_results.zip from the file browser, or run the download cell below.")
else:
    print("RUN_FORMAL_PIPELINE is False. Skipping formal training.")

## Optional Zip Download

Run the next cell in Colab if you want the packaged formal results zip to download automatically.

In [ ]:
# Optional: download packaged formal results in Colab.
from pathlib import Path

zip_path = Path("outputs/prism_formal_results.zip")
if not zip_path.exists():
    print("ERROR: results zip does not exist:", zip_path)
else:
    print(f"Zip size: {zip_path.stat().st_size / (1024 * 1024):.2f} MB")
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as exc:
        print("Not running in Colab or download is unavailable:", exc)
        print("Download manually from:", zip_path)

## Optional Stage 5 Baseline and Ablation Evaluation

Set `RUN_STAGE5_BASELINES = True` in the first cell to run Stage 5 after the formal checkpoint exists.


In [ ]:
# Optional Stage 5 baseline and ablation evaluation.
# This is disabled by default unless RUN_STAGE5_BASELINES=True in the first cell.
if RUN_STAGE5_BASELINES:
    from pathlib import Path
    import pandas as pd

    run([
        "python", "scripts/run_stage5_pipeline.py",
        "--config", "configs/toy_train.yaml",
        "--checkpoint", "outputs/checkpoints_toy/best.pt",
        "--episodes", str(BASELINE_EPISODES),
    ])

    print("\n===== Stage 5 required output check =====")
    stage5_required = [
        "outputs/results/stage5_baseline_results.csv",
        "outputs/results/stage5_baseline_episode_results.csv",
        "outputs/visualizations/stage5_baseline_comparison.png",
        "outputs/prism_stage5_results.zip",
    ]
    for output in stage5_required:
        path = Path(output)
        if path.exists():
            print(f"FOUND: {output} ({path.stat().st_size / 1024**2:.2f} MB)")
        else:
            print("MISSING:", output)

    results_csv = Path("outputs/results/stage5_baseline_results.csv")
    stage5_zip = Path("outputs/prism_stage5_results.zip")
    if results_csv.exists():
        display(pd.read_csv(results_csv))
    else:
        print("MISSING:", results_csv)

    backup_to_drive("after Stage 5 pipeline")

    if stage5_zip.exists():
        print(f"Stage 5 zip: {stage5_zip}")
        print(f"Zip size: {stage5_zip.stat().st_size / (1024 * 1024):.2f} MB")
        try:
            from google.colab import files
            files.download(str(stage5_zip))
        except Exception as exc:
            print("Not running in Colab or download is unavailable:", exc)
            print("Download manually from:", stage5_zip)
    else:
        print("MISSING:", stage5_zip)
else:
    print("Stage 5 baseline/ablation evaluation skipped. Set RUN_STAGE5_BASELINES = True to run it.")


## Save Outputs To Google Drive

Run the next cell if you want to keep checkpoints, datasets, CSV logs, and visualizations after the Colab session ends.

In [ ]:
# Optional: save outputs to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# run(['bash', '-lc', 'mkdir -p /content/drive/MyDrive/prism_outputs && cp -r outputs/* /content/drive/MyDrive/prism_outputs/'])